# Classificação: tipos de feijão (Dry Bean)

Queremos prever o tipo do feijão (sete classes) a partir de 16 medidas de tamanho e forma do grão. A análise exploratória está no notebook 03. Aqui comparamos os classificadores da `rbg`:

1. **Regressão logística**, com cinco variáveis escolhidas pela correlação.
2. **LDA e QDA**, com as 16 variáveis padronizadas.
3. **KNN** sobre as componentes principais (PCA).
4. **Árvores**: árvore com e sem poda, bagging e random forest.

Todos usam a mesma divisão estratificada (`split_dry_bean`: 10831 grãos de treino e 2712 de teste, sem as linhas duplicadas), e no fim são comparados no mesmo teste. Como as classes são desbalanceadas, além da acurácia reportamos a acurácia balanceada (média do recall das classes) e o F1 macro.

In [ ]:
import time

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

from rbg.Classification import KNN, LDA, QDA, Logistic
from rbg.DecisionTree import ClassificationTree, Bagging, RandomForest
from rbg.ModelSelection import GridSearch, KFold, StratifiedKFold
from rbg.Preprocessing import Standard
from rbg.Unsupervised import PCA
from rbg.Utilities import Accuracy, F1Score, Precision, Recall, error_rate, train_test_split

from utils.dados import (SEED, ALVO_BEAN, FEATURES_BEAN, carregar_dry_bean,
                         split_dry_bean, salvar_resultados)
from utils.avaliacao import metricas_classificacao
from utils.graficos import CLASS_COLORS, configurar_altair, stylize

configurar_altair()

## 1. Preparar os dados

Usamos a [base Dry Bean da UCI](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset), que tem sete tipos de feijão. As linhas duplicadas são removidas antes de separar treino e teste.

In [ ]:
TARGET_CLASS = ALVO_BEAN
FEATURES_CLASS = FEATURES_BEAN

dados_class = carregar_dry_bean()

display(dados_class.head())
display(dados_class[TARGET_CLASS].value_counts())

## 2. Separar treino e teste

Reservamos 20% de cada classe para o teste e usamos os 80% restantes para o treino.

In [ ]:
X_treino_completo, X_teste_completo, y_treino, y_teste = split_dry_bean(dados_class)
classes = np.unique(y_treino)

print('Treino:', X_treino_completo.shape)
print('Teste reservado:', X_teste_completo.shape)

## Parte 1: regressão logística

### 3. Escolha das variáveis

Como várias características medem coisas parecidas, vou manter cinco variáveis. Vou seguir a ordem das colunas e escolher uma variável se sua correlação absoluta com todas as já escolhidas for menor que 0,8, usando apenas o treino.

Com esse critério, vou usar `Area`, `AspectRatio`, `Extent`, `Solidity` e `Roundness`.

In [ ]:
dados_treino = pd.DataFrame(X_treino_completo, columns=FEATURES_CLASS)
correlacoes = dados_treino.corr()
colunas_escolhidas = []

for coluna in FEATURES_CLASS:
    if (correlacoes.loc[coluna, colunas_escolhidas].abs() < 0.8).all():
        colunas_escolhidas.append(coluna)
    if len(colunas_escolhidas) == 5:
        break

print(colunas_escolhidas)
display(correlacoes.loc[colunas_escolhidas, colunas_escolhidas].round(4))

indices_escolhidos = [FEATURES_CLASS.index(coluna) for coluna in colunas_escolhidas]
X_treino = X_treino_completo[:, indices_escolhidos]
X_teste = X_teste_completo[:, indices_escolhidos]

### 4. Ajustar o modelo

As variáveis têm escalas diferentes, então vou padronizá-las usando a média e o desvio do treino. Vou aplicar a mesma transformação no teste.

Vou usar o modelo com intercepto e manter `DERMASON` como referência, por ser a classe mais comum no treino.

In [ ]:
padronizador = Standard()
padronizador.fit(X_treino)
X_treino_padronizado = padronizador.transform(X_treino)
X_teste_padronizado = padronizador.transform(X_teste)

referencia = 'DERMASON'
modelo_final = Logistic(
    classes=classes, reference_class=referencia, fit_intercept=True
)
modelo_final.fit(X_treino_padronizado, y_treino)
print('Referência:', referencia)

### 5. Avaliação no teste

Vou avaliar o modelo com acurácia e acurácia balanceada. Como as classes têm quantidades diferentes de observações, a balanceada permite dar o mesmo peso ao recall de cada uma. Também vou comparar precisão e recall por classe.

In [ ]:
previsoes_teste = modelo_final.predict(X_teste_padronizado)
recall_teste = Recall(previsoes_teste, y_teste)

print(f'Acurácia: {Accuracy(previsoes_teste, y_teste):.4f}')
print(f'Acurácia balanceada: {recall_teste.mean():.4f}')

metricas_classes_teste = pd.DataFrame({
    'classe': classes,
    'precisao': Precision(previsoes_teste, y_teste),
    'recall': recall_teste
})
display(metricas_classes_teste.round(4))

resultado_logistica = metricas_classificacao('Logística (5 variáveis)', previsoes_teste, y_teste)

### 6. Inferência

Vou usar `k_coef()` para mostrar os betas de cada classe em relação a `DERMASON`. Cada cor representa uma variável e a altura da barra mostra seu beta estimado.

Vou deixar o intercepto fora do gráfico e não mostrar `BOMBAY`, por ter menos amostras. Ela continua no modelo e nas métricas.

In [ ]:
classes_inferencia = [classe for classe in classes if classe not in [referencia, 'BOMBAY']]

tabelas = []
for classe in classes_inferencia:
    tabela = pd.DataFrame({
        'classe': classe,
        'variavel': colunas_escolhidas,
        'beta': modelo_final.k_coef(classe)[1:]
    })
    tabelas.append(tabela)

tabela_inferencia = pd.concat(tabelas, ignore_index=True)

alt.Chart(tabela_inferencia).mark_bar().encode(
    x=alt.X('classe:N', title='Tipo de feijão', axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset('variavel:N', sort=colunas_escolhidas),
    y=alt.Y('beta:Q', title='Beta — referência: DERMASON'),
    color=alt.Color('variavel:N', title='Variável', sort=colunas_escolhidas),
    tooltip=['classe:N', 'variavel:N', alt.Tooltip('beta:Q', format='.4f')]
).properties(width=700, height=350)

Com essas barras, conseguimos notar como cada variável está associada à chance de cada tipo de feijão em relação a `DERMASON`, mantendo as outras variáveis fixas. Por exemplo, um `AspectRatio` maior favorece `HOROZ`, enquanto um valor menor favorece `SEKER`, nessa comparação.

A `Area` tem beta positivo em todas as classes mostradas: grãos maiores favorecem essas classes em relação a `DERMASON`. Já `Roundness` tem beta negativo para `BARBUNYA` e positivo para `CALI`, indicando associações em sentidos diferentes.

As barras representam mudanças no logaritmo da razão entre as probabilidades da classe e de `DERMASON`, não aumentos diretos de probabilidade.

### 7. Comentários sobre os resultados

Com a mesma divisão estratificada, o modelo teve acurácia de cerca de 91,7% no teste. `BOMBAY`, a classe mais distinta, é identificada corretamente em praticamente todos os casos, enquanto `SIRA` e `BARBUNYA` têm os menores valores de recall, por se confundirem com `DERMASON` e `CALI`.

A escolha por correlação diminui a redundância, mas não permite concluir que essas sejam as cinco melhores variáveis.

## Parte 2: LDA e QDA

A LDA e a QDA modelam cada classe como uma normal multivariada e classificam pela maior probabilidade a posteriori. A LDA supõe a mesma matriz de covariância em todas as classes, o que dá fronteiras lineares; a QDA estima uma covariância por classe, o que dá fronteiras quadráticas e mais parâmetros. Aqui usamos as 16 variáveis padronizadas com o `Standard` ajustado no treino. Os dois modelos não têm hiperparâmetros para escolher, então vão direto ao teste.

In [ ]:
padronizador_16 = Standard()
padronizador_16.fit(X_treino_completo)
Z_treino = padronizador_16.transform(X_treino_completo)
Z_teste = padronizador_16.transform(X_teste_completo)

resultados_discriminantes = []
previsoes_discriminantes = {}
for nome, modelo in [('LDA', LDA(labels=classes)), ('QDA', QDA(labels=classes))]:
    inicio = time.time()
    modelo.fit(Z_treino, y_treino)
    previsoes_discriminantes[nome] = modelo.predict(Z_teste)
    resultado = metricas_classificacao(f'{nome} (16 variáveis)', previsoes_discriminantes[nome], y_teste)
    resultado['tempo_s'] = round(time.time() - inicio, 1)
    resultados_discriminantes.append(resultado)

pd.DataFrame(resultados_discriminantes).round(4)

In [ ]:
recall_por_classe = pd.DataFrame({
    'classe': classes,
    'recall LDA': Recall(previsoes_discriminantes['LDA'], y_teste),
    'recall QDA': Recall(previsoes_discriminantes['QDA'], y_teste),
})
recall_por_classe.round(4)

A comparação entre as duas mostra se vale a pena pagar pelos parâmetros extras da QDA: se ela acerta mais, as classes têm formas (covariâncias) diferentes o suficiente para que fronteiras curvas ajudem. Pela tabela de recall, as classes mais difíceis continuam sendo `SIRA`, `DERMASON` e `BARBUNYA`, as mesmas da logística.

## Parte 3: KNN com PCA

### Pré-processamento

O PCA e o KNN dependem da escala das variáveis: sem padronizar, `Area` (dezenas de milhares) dominaria as distâncias e as componentes, e `Solidity` (entre 0 e 1) quase não contaria. A média e o desvio são calculados só no treino e aplicados ao teste, para que o teste não influencie o modelo. Reaproveitamos a padronização das 16 variáveis feita para a LDA e a QDA.

### Redução de dimensionalidade

Várias variáveis medem quase a mesma coisa (`Area`, `Perimeter`, `ConvexArea`, `EquivDiameter`...). O PCA troca essas variáveis correlacionadas por componentes ortogonais, e com menos dimensões as distâncias do KNN ficam mais informativas e mais rápidas de calcular. Mantemos o número de componentes necessário para reter 95% da variância do treino.

In [ ]:
pca = PCA(n_components=0.95)  # variância acumulada desejada
pca.fit(Z_treino)

X_train_pca = pca.transform(Z_treino)
X_test_pca = pca.transform(Z_teste)

print(f"Dimensão original das features: {Z_treino.shape[1]}")
print(f"Componentes mantidos pelo PCA: {pca.components.shape[0]}")
print(f"Variância acumulada: {np.sum(pca.explained_variance_ratio) * 100:.2f}%")

Com 4 componentes a variância acumulada fica em 94,98%, logo abaixo do corte, então o PCA mantém 5 componentes (cerca de 97,7% da variância). Mesmo assim, passamos de 16 para 5 dimensões.

### Escolha do k

O $k$ é escolhido por `GridSearch` com `StratifiedKFold` (5 partes, mantendo a proporção das classes), entre valores ímpares para evitar empates na votação, maximizando o F1 macro, que é mais adequado do que a acurácia com classes desbalanceadas.

Como o KNN da `rbg` calcula a distância para todos os pontos de treino a cada previsão, a validação cruzada no treino inteiro seria lenta. Por isso a busca é feita em uma subamostra aleatória de 1000 grãos do treino (com a semente do projeto). Depois o modelo com o melhor $k$ é ajustado no treino completo e avaliado uma única vez no teste.

In [ ]:
_, X_mini, _, y_mini = train_test_split(X_train_pca, y_treino, test_size=1000, random_state=SEED)

def f1_macro(previsto, real):
    return F1Score(previsto, real).mean()

valores_de_k = [3, 5, 7, 9, 11, 15]
grid = GridSearch(
    model=KNN,
    param_grid={'k': valores_de_k},
    metrics=f1_macro,
    greater_is_better=True,
    cv=StratifiedKFold(k=5, shuffle=True, random_state=SEED)
)

grid.fit(X_mini, y_mini)
best_k = grid.best_params_['k']
print("Melhor hiperparâmetro encontrado:", best_k)
print(f"F1 macro médio na validação cruzada: {grid.best_score_ * 100:.2f}%")

### Avaliação final do KNN

In [ ]:
inicio = time.time()
best_knn = KNN(k=best_k)
best_knn.fit(X_train_pca, y_treino)
y_pred_knn = best_knn.predict(X_test_pca)

resultado_knn = metricas_classificacao(f'KNN (k = {best_k}, PCA com {X_train_pca.shape[1]} componentes)', y_pred_knn, y_teste)
print(f"tempo de previsão do teste: {time.time() - inicio:.0f} s")
pd.DataFrame([resultado_knn]).round(4)

### Visualização em duas componentes

Para enxergar por que o KNN erra, projetamos o treino nas duas primeiras componentes principais.

In [ ]:
pca2d = PCA(n_components=2)
pca2d.fit(Z_treino)
X_train_pca2d = pca2d.transform(Z_treino)
print(f"Variância acumulada nas duas componentes: {np.sum(pca2d.explained_variance_ratio) * 100:.2f}%")

df_pca_plot = pd.DataFrame({"PC1": X_train_pca2d[:, 0], "PC2": X_train_pca2d[:, 1], "Class": y_treino})
color_scale = alt.Scale(domain=list(CLASS_COLORS.keys()), range=list(CLASS_COLORS.values()))

stylize(
    alt.Chart(df_pca_plot)
    .mark_circle(size=20, opacity=0.6)
    .encode(
        x=alt.X("PC1:Q", title="Componente Principal 1 (PC1)"),
        y=alt.Y("PC2:Q", title="Componente Principal 2 (PC2)"),
        color=alt.Color("Class:N", scale=color_scale, title="Classe"),
    )
    .properties(title="Dispersão das amostras de treino nas duas primeiras componentes", width=650, height=450)
)

A projeção nas duas primeiras componentes mostra por que o KNN erra: `BOMBAY` fica isolada, mas `CALI` e `BARBUNYA` se sobrepõem, e `SIRA` fica em uma região de fronteira com `DERMASON` e `SEKER`. Nessas regiões, os vizinhos mais próximos de um grão costumam ser de outra classe. As duas componentes são só uma visualização: o modelo usa as 5 componentes.

## Parte 4: árvores

Faremos agora o uso de árvores para prever os nossos dados: árvores de classificação com e sem poda, bagging e random forest. Como no KNN, usamos as componentes do PCA (`X_train_pca` e `X_test_pca`) em vez das 16 colunas originais: com menos colunas, a árvore tem menos variáveis em que procurar o ponto de corte.

* **Validação**: para árvores, random forest e bagging, usaremos o GridSearch para procurar, em cada um dos três, os melhores hiperparâmetros.
* **Comparação de modelos**: com os modelos ajustados pelos seus melhores hiperparâmetros, faremos uma validação cruzada com os dados de treino para decidir qual o melhor modelo.
* **Avaliação final**: dado o modelo escolhido, faremos a predição dos dados de teste e calcularemos seu erro (`error_rate`, que é 1 menos a acurácia).
* **Interpretação dos resultados**: considerações sobre a escolha das métricas e o trade-off entre viés e variância.

### Tamanho da amostra

A construção das árvores na `rbg` testa todos os valores de cada variável como ponto de corte em cada nó, e o tempo cresce rápido com o número de linhas: uma árvore com 4000 grãos já leva perto de um minuto, e o treino tem 10831. Com GridSearch e K-Fold isso seria multiplicado por dezenas de ajustes. Por isso, em toda esta parte as árvores são treinadas em uma subamostra aleatória de 1500 grãos do treino (sorteada com a semente do projeto). O teste continua sendo o mesmo, com os 2712 grãos.

In [ ]:
rng = np.random.default_rng(SEED)
indices_sub = rng.choice(len(y_treino), size=1500, replace=False)
X_sub = X_train_pca[indices_sub]
y_sub = y_treino[indices_sub]

pd.DataFrame({'treino completo': pd.Series(y_treino).value_counts(normalize=True),
              'subamostra': pd.Series(y_sub).value_counts(normalize=True)}).round(3)

In [ ]:
inicio = time.time()
arvore = ClassificationTree()

arvore.fit(X_sub, y_sub)

y_previsto = arvore.predict(X_test_pca)

erro_arvore_sem_poda = error_rate(y_previsto, y_teste)

print("erro da árvore sem poda:", erro_arvore_sem_poda)
print(f"tempo: {time.time() - inicio:.0f} s")

In [ ]:
inicio = time.time()
arvore_poda = ClassificationTree()

arvore_poda.fit(X_sub, y_sub, prune=True)

y_previsto_poda = arvore_poda.predict(X_test_pca)

erro_arvore_com_poda = error_rate(y_previsto_poda, y_teste)

print("erro da árvore com poda:", erro_arvore_com_poda)
print(f"tempo: {time.time() - inicio:.0f} s")

Com o padrão de 100 árvores, o bagging com 5000 linhas levou mais de 60 minutos nos nossos testes. Para ser computacionalmente viável, tanto o random forest quanto o bagging usam apenas 2 ou 3 árvores nesta parte.

In [ ]:
inicio = time.time()
arvore_random = RandomForest(n_trees=3)

arvore_random.fit(X_sub, y_sub)

y_previsto_random = arvore_random.predict(X_test_pca)

erro_arvore_random = error_rate(y_previsto_random, y_teste)

print("erro do random forest:", erro_arvore_random)
print(f"tempo: {time.time() - inicio:.0f} s")

In [ ]:
inicio = time.time()
arvorebag = Bagging(n_trees=3)

arvorebag.fit(X_sub, y_sub)

y_previstobag = arvorebag.predict(X_test_pca)

erro_arvore_bag = error_rate(y_previstobag, y_teste)

print("erro do bagging:", erro_arvore_bag)
print(f"tempo: {time.time() - inicio:.0f} s")

### Validação

Uma vez que árvores (até mesmo as mais simples, sem poda e sem bagging) têm um custo grande para serem montadas de acordo com o algoritmo implementado, e uma vez que as árvores possuem vários parâmetros, então para fazer uma validação otimizada usaremos apenas os hiperparâmetros:

- min_samples_split (e não usaremos min_samples_leaf, que ficará padronizado como 1)
- max_depth (e não usaremos max_leaf)
- min_impurity_decrease
- criterion_split ("Standard" é o índice de Gini; a outra opção é a entropia)

Dentro de criterion_choose, iremos avaliar apenas a moda, já que a média não faz sentido em um problema de classificação, e a mediana também não, porque os tipos de feijão não têm uma ordem. Também deixaremos de fora "min" e "max". A métrica de validação é o `error_rate`.

Todas as validações usam K-Fold com 5 partes, embaralhado com a semente do projeto.

In [ ]:
p_grid = {
    'max_depth': [5],
    'min_samples_split': [10, 50],
    'min_impurity_decrease': [0.1],
    'criterion_split': ['Standard', 'entropy'],
    'criterion_choose': ['Standard']
}

grid_arvore = GridSearch(model=ClassificationTree(), param_grid=p_grid, metrics=error_rate,
                         cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_arvore.fit(X_sub, y_sub)
print(f"GridSearch da árvore: {time.time() - inicio:.0f} s")
print(grid_arvore.best_params_)
print("erro médio na validação:", grid_arvore.best_score_)

Para o bagging, que constrói as árvores uma de cada vez (sem paralelização), testamos menos combinações ainda e variamos o número de árvores entre 2 e 3.

In [ ]:
pbaggin_grid = {
    'max_depth': [7],
    'min_samples_split': [10],
    'min_impurity_decrease': [0.1],
    'criterion_split': ['entropy'],
    'criterion_choose': ['Standard'],
    'n_trees': [3, 2]
}

grid_bagging = GridSearch(model=Bagging(), param_grid=pbaggin_grid, metrics=error_rate,
                          cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_bagging.fit(X_sub, y_sub)
print(f"GridSearch do bagging: {time.time() - inicio:.0f} s")
print(grid_bagging.best_params_)
print("erro médio na validação:", grid_bagging.best_score_)

Similarmente, para o random forest testamos 2 e 3 árvores, com o número de variáveis sorteadas em cada corte igual ao piso da raiz quadrada do total. Como usamos as componentes do PCA, temos 5 variáveis e, portanto, 2 são sorteadas em cada corte.

In [ ]:
p_random_grid = {
    'max_depth': [7],
    'min_samples_split': [10],
    'min_impurity_decrease': [0.1],
    'criterion_split': ['entropy'],
    'criterion_choose': ['Standard'],
    'n_trees': [3, 2],
    'factor_selection': [np.sqrt]
}

grid_Random = GridSearch(model=RandomForest(), param_grid=p_random_grid, metrics=error_rate,
                         cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_Random.fit(X_sub, y_sub)
print(f"GridSearch do random forest: {time.time() - inicio:.0f} s")
print(grid_Random.best_params_)
print("erro médio na validação:", grid_Random.best_score_)

### Seleção de modelo

Para a seleção de modelos, faremos um K-Fold usando os campeões de cada GridSearch: treinamos cada um nas partes de treino e escolhemos aquele que teve o menor `error_rate` médio. Usamos uma semente diferente da do GridSearch para que as partes não sejam as mesmas que escolheram os hiperparâmetros.

In [ ]:
param_r = grid_Random.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

bagging = Bagging(**param_bagging)
arvore_pos_grid = ClassificationTree(**param_arvore)
random_forest = RandomForest(**param_r)

In [ ]:
inicio = time.time()
k_fold = KFold(k=5, shuffle=True, random_state=SEED + 1)
lista_erros_arvore = []
lista_erros_bag = []
lista_erros_random = []

for treino_k, teste_k in k_fold.split(X_sub):
    X_treino_k, X_teste_k = X_sub[treino_k], X_sub[teste_k]
    y_treino_k, y_teste_k = y_sub[treino_k], y_sub[teste_k]

    # árvore nova a cada parte: refazer o fit no mesmo objeto de árvore da rbg
    # reaproveita estado do ajuste anterior e piora a árvore
    arvore_pos_grid = ClassificationTree(**param_arvore)
    random_forest.fit(X_treino_k, y_treino_k)
    bagging.fit(X_treino_k, y_treino_k)
    arvore_pos_grid.fit(X_treino_k, y_treino_k)
    lista_erros_arvore.append(error_rate(arvore_pos_grid.predict(X_teste_k), y_teste_k))
    lista_erros_bag.append(error_rate(bagging.predict(X_teste_k), y_teste_k))
    lista_erros_random.append(error_rate(random_forest.predict(X_teste_k), y_teste_k))

erros_kfold = pd.DataFrame({
    'modelo': ['árvore', 'bagging', 'random forest'],
    'erro médio': [np.mean(lista_erros_arvore), np.mean(lista_erros_bag), np.mean(lista_erros_random)],
})
print(f"tempo: {time.time() - inicio:.0f} s")
erros_kfold.round(4)

### Avaliação final

Com o modelo ganhador, treinamos na subamostra inteira e calculamos o erro no teste. Os outros dois também são ajustados, para entrar na tabela comparativa.

In [ ]:
arvore_pos_grid = ClassificationTree(**param_arvore)  # objeto novo para o ajuste final
modelos_arvore = {'árvore': arvore_pos_grid, 'bagging': bagging, 'random forest': random_forest}
vencedor = erros_kfold.sort_values('erro médio').iloc[0]['modelo']
print("vencedor:", vencedor)

previsoes_arvores = {}
for nome, modelo in modelos_arvore.items():
    modelo.fit(X_sub, y_sub)
    previsoes_arvores[nome] = modelo.predict(X_test_pca)

print("Erro do modelo no teste:", error_rate(previsoes_arvores[vencedor], y_teste))

O bagging guarda quais grãos ficaram fora de cada amostra bootstrap, o que permite calcular o erro OOB (out-of-bag) sem usar o teste. O método `OOB()` da `rbg` converte as previsões para número e por isso só funciona quando as classes são números; aqui as classes são nomes, então fazemos a mesma conta usando os atributos que o bagging guarda (`list_of_trees` e `list_of_indx_test`).

In [ ]:
previsoes_oob, reais_oob = [], []
for i in range(len(y_sub)):
    arvores_fora = [arv for arv, fora in zip(bagging.list_of_trees, bagging.list_of_indx_test) if i in fora]
    if arvores_fora:
        previsoes_oob.append(bagging.predict(X_sub[i:i + 1], trees=arvores_fora)[0])
        reais_oob.append(y_sub[i])

print(f"grãos com previsão OOB: {len(reais_oob)} de {len(y_sub)}")
print("erro OOB do bagging:", error_rate(np.array(previsoes_oob), np.array(reais_oob)))

### Interpretação dos resultados

Faz sentido o bagging vencer a árvore única: mesmo com poucas árvores, a média dos votos diminui a variância. O random forest, com apenas 2 ou 3 árvores e só 2 das 5 componentes sorteadas em cada corte, tende a ficar pior, porque cada árvore individual é mais fraca e não há árvores suficientes para compensar. Além disso, como o PCA já concentrou a informação nas primeiras componentes, sortear variáveis pode deixar de fora justamente a componente mais útil.

A principal limitação desta parte é o tamanho da amostra: as árvores usam 1500 dos 10831 grãos de treino, enquanto os outros modelos usam o treino inteiro. Por isso a comparação com eles é desfavorável às árvores.

## Comparação final no teste

Todos os modelos foram escolhidos usando só o treino; aqui medimos cada um no mesmo teste de 2712 grãos. A rede neural está no notebook 05 e entra na comparação do relatório final.

In [ ]:
comparacao = [resultado_logistica, *[{k: v for k, v in r.items() if k != 'tempo_s'} for r in resultados_discriminantes], resultado_knn]
comparacao.append(metricas_classificacao('Árvore com poda (padrão, 1500 grãos)', y_previsto_poda, y_teste))
comparacao.append(metricas_classificacao('Árvore (GridSearch, 1500 grãos)', previsoes_arvores['árvore'], y_teste))
comparacao.append(metricas_classificacao(f"Bagging ({param_bagging['n_trees']} árvores, 1500 grãos)", previsoes_arvores['bagging'], y_teste))
comparacao.append(metricas_classificacao(f"Random forest ({param_r['n_trees']} árvores, 1500 grãos)", previsoes_arvores['random forest'], y_teste))

tabela_teste = pd.DataFrame(comparacao).sort_values('acuracia', ascending=False).reset_index(drop=True)
tabela_teste.round(4)

In [ ]:
salvar_resultados("classificacao", tabela_teste)

## Fechamento

A tabela acima reúne todos os classificadores no mesmo teste; a rede neural (notebook 05) entra na comparação do relatório final. Em todos os modelos os erros se concentram nos pares que a análise exploratória já apontava, `DERMASON` × `SIRA` e `CALI` × `BARBUNYA`. A logística é a mais interpretável (cada beta tem leitura direta) e treina rápido; LDA e QDA também não têm hiperparâmetros; o KNN precisa da padronização, do PCA e da escolha do k, e é lento para prever porque compara cada grão com todo o treino; as árvores da `rbg` são as mais caras e, por isso, foram treinadas em uma subamostra.